<a href="https://colab.research.google.com/github/Lupo6789/tesi_scacchi/blob/main/02_Feature_Engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
import chess
import pandas as pd

#Caricamento e Filtro Lunghezza
df = pd.read_csv('games_cleaned.csv')
df['numero_mosse_reali'] = df['moves'].apply(lambda x: len(x.split()))
df = df[df['numero_mosse_reali'] >= 40].copy()
df = df.drop(columns=['numero_mosse_reali'])
print(f"Partite valide (almeno 40 semimosse): {df.shape[0]}\n")

#Funzione unica per TUTTE le feature posizionali
def estrai_features_posizionali(moves_string, n_semimosse):
    board = chess.Board()
    mosse = moves_string.split()

    for i in range(min(n_semimosse, len(mosse))):
        board.push_san(mosse[i])

    #Materiale
    valori = {chess.PAWN: 1, chess.KNIGHT: 3, chess.BISHOP: 3, chess.ROOK: 5, chess.QUEEN: 9}
    mat_b = sum(valori.get(p.piece_type, 0) for p in board.piece_map().values() if p.color == chess.WHITE)
    mat_n = sum(valori.get(p.piece_type, 0) for p in board.piece_map().values() if p.color == chess.BLACK)
    vantaggio_materiale = mat_b - mat_n

    # Mobilità
    turno_originale = board.turn
    board.turn = chess.WHITE
    mob_b = board.legal_moves.count()
    board.turn = chess.BLACK
    mob_n = board.legal_moves.count()
    board.turn = turno_originale
    vantaggio_mobilita = mob_b - mob_n

    #Arrocco e Scacco
    arr_b = int(board.has_castling_rights(chess.WHITE))
    arr_n = int(board.has_castling_rights(chess.BLACK))
    scacco = 1 if board.is_check() else 0

    # Sviluppo Pezzi Leggeri
    # Contiamo Cavalli e Alfieri che NON sono più nelle case di partenza, per gli altri  pezzi non serve
    sviluppo_b = sum(1 for sq in board.pieces(chess.KNIGHT, chess.WHITE) if sq not in [chess.B1, chess.G1]) + \
                 sum(1 for sq in board.pieces(chess.BISHOP, chess.WHITE) if sq not in [chess.C1, chess.F1])

    sviluppo_n = sum(1 for sq in board.pieces(chess.KNIGHT, chess.BLACK) if sq not in [chess.B8, chess.G8]) + \
                 sum(1 for sq in board.pieces(chess.BISHOP, chess.BLACK) if sq not in [chess.C8, chess.F8])

    vantaggio_sviluppo = sviluppo_b - sviluppo_n

    return vantaggio_materiale, vantaggio_mobilita, arr_b, arr_n, scacco, vantaggio_sviluppo

#Estrazione ai 4 momenti chiave
soglie = [10, 20, 30, 40]

for N in soglie:
    print(f"Estrazione in corso per N={N} semimosse")
    risultati = df['moves'].apply(lambda x: estrai_features_posizionali(x, N))

    #Creo 6 nuove colonne per ogni step (totale 24 nuove variabili)
    col_names = [f'mat_{N}', f'mob_{N}', f'arr_b_{N}', f'arr_n_{N}', f'scacco_{N}', f'sviluppo_{N}']
    df[col_names] = pd.DataFrame(risultati.tolist(), index=df.index)

#Salvataggio
df.to_csv('games_features_complete.csv', index=False)
print("\n FASE POSIZIONALE COMPLETATA AL 100%")
print("File 'games_features_complete.csv' salvato.")

Partite valide (almeno 40 semimosse): 13007

Estrazione in corso per N=10 semimosse
Estrazione in corso per N=20 semimosse
Estrazione in corso per N=30 semimosse
Estrazione in corso per N=40 semimosse

 FASE POSIZIONALE COMPLETATA AL 100%
File 'games_features_complete.csv' salvato.


In [14]:
import pandas as pd

# 1. Carichiamo il dataset con tutte le 24 feature posizionali appena create
df = pd.read_csv('games_features_complete.csv')

# 2. Calcolo Feature Pre-partita (Rating, ECO, Tempo)
df['rating_diff'] = df['white_rating'] - df['black_rating']
df['rating_mean'] = (df['white_rating'] + df['black_rating']) / 2
df['eco_group'] = df['opening_eco'].astype(str).str[0]
df['base_time_min'] = df['increment_code'].apply(lambda x: int(x.split('+')[0]) if '+' in str(x) else 0)

# 3. Salvataggio Dataset Definitivo
df.to_csv('games_features_full.csv', index=False)

print("FASE PRE-PARTITA COMPLETATA")
print("Nuove colonne aggiunte: 'rating_diff', 'rating_mean', 'eco_group', 'base_time_min'.")
print("Il file 'games_features_full.csv' in teoria va bene")

FASE PRE-PARTITA COMPLETATA
Nuove colonne aggiunte: 'rating_diff', 'rating_mean', 'eco_group', 'base_time_min'.
Il file 'games_features_full.csv' in teoria va bene
